# Feature Engineering M5 Forecasting Accuracy (Polars, chunked per-store)

Versi ini mengatasi masalah baru yang muncul setelah migrasi ke Polars untuk `MODE="full"`:
mesin yang dipakai punya **16 GB RAM (~10 GB usable)**, sementara `combined_df` penuh
(~58 juta baris x ~90 kolom setelah semua fitur ditambahkan) butuh memory jauh lebih
besar dari itu. Begitu usable RAM habis, Windows mulai swap ke disk, dan operasi
sesederhana `filter` bisa tiba-tiba butuh berjam-jam (bukan salah kode, murni karena
data sudah tidak muat di RAM).

**Penting**: `pl.read_csv` / `pl.scan_csv` pada file `.csv.gz` selalu mendekompresi
seluruh file ke memory dulu sebelum diproses — belum ada mode streaming untuk sumber
gzip di Polars (per Juli 2026). Jadi trik "lazy scan + filter per store" TIDAK
menghemat memory sama sekali untuk file `.csv.gz` — file tetap didekompresi penuh
di awal, hanya lebih banyak kali kalau dilakukan per-store.

## Strategi baru: partisi sekali (pakai pandas chunked reader), lalu proses per-store (pakai Polars)

1. **Fase 1 (partisi, hemat memory)** - baca `train_preprocessed_full.csv.gz` dan
   `test_preprocessed_full.csv.gz` dengan `pandas.read_csv(..., chunksize=...)`.
   Pandas benar-benar men-stream file gzip per beberapa juta baris saja, sehingga
   memory yang dipakai di fase ini konstan kecil, tidak peduli seberapa besar file
   totalnya. Setiap chunk langsung difan-out ke 1 file Parquet per `store_id`
   (append per chunk via `pyarrow.parquet.ParquetWriter`). Statistik `global_price_mean`
   juga diakumulasi di fase ini (sekali jalan, murah).
2. **Fase 2 (feature engineering, per store, pakai Polars)** - untuk setiap store
   (M5 full = 10 store), baca hanya file Parquet store itu (~1/10 ukuran total),
   bangun semua fitur yang sama seperti sebelumnya (calendar, price, lag, rolling),
   lalu **append** hasilnya ke satu file gabungan `train_features_full.csv.gz` /
   `test_features_full.csv.gz` (stream gzip yang sama dibuka sekali di awal loop dan
   ditutup di akhir - bukan dibuka-tutup per store, supaya tidak jadi multi-member
   gzip yang berisiko tidak semua reader mendukungnya).
3. Hasil akhirnya **tetap satu dataset gabungan**, identik strukturnya dengan versi
   sebelumnya - loop per-store murni trik internal supaya peak memory ~1/10 dari
   sebelumnya, bukan perubahan pada kontrak file output.

## Fitur yang dibangun (tidak berubah)
- Lag features: `7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 90, 180, 364`
- Rolling statistics: `mean`, `std`, `max`, `min` untuk window `7, 14, 28, 60, 90, 180, 364`
- Calendar features: `week`, `day`, `days_to_event` (event calendar sama untuk semua store, jadi aman dihitung per-store)
- Price features: `price_mean`, `price_norm`, `price_cat_avg`, `price_cat_ratio`, `price_change`, `price_changed`, `is_discounted`

## Yang tetap terjaga vs. yang berbeda dari versi single-pass sebelumnya
- **Tidak ada `id` yang terpotong lintas store**: format `id` M5 menyertakan `store_id`
  (contoh: `HOBBIES_1_001_CA_1_validation`), jadi setiap seri item-store utuh berada
  di satu partisi store. Fitur `lag`/`rolling` dihitung `.over("id")`, jadi hasilnya
  identik dengan versi single-pass selama urutan tanggal dalam id tetap terjaga (sudah dijamin oleh `.sort(["id","date"])` per store).
- **price_mean / price_cat_avg** dikelompokkan per `store_id` sejak awal, jadi menghitungnya per-store memberi hasil yang sama persis dengan versi single-pass. `global_price_mean` diakumulasi sekali dari seluruh train di Fase 1, juga tidak berubah.
- **Tidak ada baris yang hilang/dobel** - dicek otomatis lewat assertion jumlah baris setelah loop selesai.
- **Yang beda: urutan baris di file akhir.** Versi sebelumnya mengurutkan seluruh data
  sekaligus by `id, date` (yang, karena `item_id` muncul sebelum `store_id` di string
  `id`, membuat store-store saling berselang-seling per item). Versi ini menulis
  per-store, jadi hasil akhirnya terkelompok per store dulu, baru per `id, date` di
  dalamnya. Ini tidak masalah untuk model berbasis tree (LightGBM/XGBoost/CatBoost)
  yang tidak peduli urutan baris, tapi kalau notebook modeling nanti mengandalkan
  urutan/posisi baris (bukan filter berdasarkan kolom tanggal/split), ini perlu
  diperhatikan.
- Ada juga pengecekan otomatis bahwa urutan kolom identik di setiap store sebelum
  ditulis tanpa header, supaya tidak ada risiko kolom "geser" di bawah nama header
  yang salah.

## Catatan penting untuk langkah berikutnya
File output akhir (`train_features_full.csv.gz`) tetap berukuran penuh (semua store
digabung). Kalau notebook modeling berikutnya membacanya dengan `pd.read_csv` /
`pl.read_csv` sekaligus, masalah memory yang sama bisa muncul lagi di sana. Sel
paling akhir notebook ini juga menulis salinan `.parquet` (partitioned per store)
yang jauh lebih kecil dan lebih cepat dibaca sebagian-sebagian - pertimbangkan
memakai itu di notebook modeling kalau training juga perlu dilakukan per-store atau
per-fold, bukan memuat semuanya sekaligus.


In [1]:
from __future__ import annotations

import gc
import gzip
import json
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

pl.Config.set_tbl_cols(60)
pl.Config.set_tbl_width_chars(180)
print("polars:", pl.__version__)


polars: 1.43.0


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
PREPROCESSED_DIR = DATASET_DIR / "preprocessed"
FEATURE_DIR = DATASET_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

MODE = "full"
COMPRESSION = "gzip"
LAG_DAYS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 90, 180, 364]
ROLLING_WINDOWS = [7, 14, 28, 60, 90, 180, 364]
ROLLING_STATS = ["mean", "std", "max", "min"]

# Rows read per pandas chunk while partitioning by store. Lower this (e.g. 500_000)
# if Fase 1 itself still uses too much memory on your machine.
PARTITION_CHUNK_ROWS = 2_000_000

TRAIN_INPUT_PATH = PREPROCESSED_DIR / f"train_preprocessed_{MODE}.csv.gz"
TEST_INPUT_PATH = PREPROCESSED_DIR / f"test_preprocessed_{MODE}.csv.gz"
PREPROCESSING_MAPPINGS_PATH = PREPROCESSED_DIR / f"preprocessing_mappings_{MODE}.json"
PREPROCESSING_SUMMARY_PATH = PREPROCESSED_DIR / f"preprocessing_summary_{MODE}.json"

TRAIN_PARTITION_DIR = PREPROCESSED_DIR / f"_tmp_partitioned_{MODE}_train"
TEST_PARTITION_DIR = PREPROCESSED_DIR / f"_tmp_partitioned_{MODE}_test"

TRAIN_OUTPUT_PATH = FEATURE_DIR / f"train_features_{MODE}.csv.gz"
TEST_OUTPUT_PATH = FEATURE_DIR / f"test_features_{MODE}.csv.gz"
TRAIN_OUTPUT_PARQUET_DIR = FEATURE_DIR / f"train_features_{MODE}_by_store"
TEST_OUTPUT_PARQUET_DIR = FEATURE_DIR / f"test_features_{MODE}_by_store"
SUMMARY_OUTPUT_PATH = FEATURE_DIR / f"feature_engineering_summary_{MODE}.json"

assert TRAIN_INPUT_PATH.exists(), f"Input train preprocessing tidak ditemukan: {TRAIN_INPUT_PATH}"
assert TEST_INPUT_PATH.exists(), f"Input test preprocessing tidak ditemukan: {TEST_INPUT_PATH}"

print(f"PROJECT_ROOT:      {PROJECT_ROOT}")
print(f"MODE:              {MODE}")
print(f"TRAIN_INPUT_PATH:  {TRAIN_INPUT_PATH}")
print(f"TEST_INPUT_PATH:   {TEST_INPUT_PATH}")
print(f"TRAIN_OUTPUT_PATH: {TRAIN_OUTPUT_PATH}")
print(f"TEST_OUTPUT_PATH:  {TEST_OUTPUT_PATH}")


PROJECT_ROOT:      C:\Users\Lenovo\Documents\CODING\Python\speedrun-lulus
MODE:              full
TRAIN_INPUT_PATH:  C:\Users\Lenovo\Documents\CODING\Python\speedrun-lulus\dataset\preprocessed\train_preprocessed_full.csv.gz
TEST_INPUT_PATH:   C:\Users\Lenovo\Documents\CODING\Python\speedrun-lulus\dataset\preprocessed\test_preprocessed_full.csv.gz
TRAIN_OUTPUT_PATH: C:\Users\Lenovo\Documents\CODING\Python\speedrun-lulus\dataset\features\train_features_full.csv.gz
TEST_OUTPUT_PATH:  C:\Users\Lenovo\Documents\CODING\Python\speedrun-lulus\dataset\features\test_features_full.csv.gz


## Fase 1 - partisi per store (pandas chunked reader, memory konstan kecil)

`pandas.read_csv(..., chunksize=...)` benar-benar men-stream file gzip: setiap
iterasi hanya memegang `PARTITION_CHUNK_ROWS` baris di memory, lalu dibuang begitu
selesai diproses. Ini satu-satunya cara aman membaca file gzip besar tanpa
mendekompresi semuanya sekaligus (yang menjadi masalah utama di versi sebelumnya).

Dtype numerik dikunci lewat parameter `dtype=` supaya konsisten di semua chunk
(pandas bisa diam-diam mengubah int jadi float kalau ada NaN di chunk tertentu tapi
tidak di chunk lain, yang akan merusak penulisan Parquet per-store kalau tidak
dikunci dari awal).


In [3]:
PANDAS_DTYPE_MAP = {
    "wm_yr_wk": "int32",
    "wday": "int8",
    "dayofweek": "int8",
    "month": "int8",
    "year": "int16",
    "is_weekend": "int8",
    "snap": "int8",
    "sales": "int32",
    "event_name_1_le": "int16",
    "event_name_2_le": "int16",
    "item_id_le": "int16",
    "dept_id_le": "int8",
    "cat_id_le": "int8",
    "store_id_le": "int8",
    "state_id_le": "int8",
    "store_cat_le": "int8",
    "store_dept_le": "int8",
    "event_type_1__Cultural": "int8",
    "event_type_1__National": "int8",
    "event_type_1__No_Type": "int8",
    "event_type_1__Religious": "int8",
    "event_type_1__Sporting": "int8",
    "event_type_2__Cultural": "int8",
    "event_type_2__No_Type": "int8",
    "event_type_2__Religious": "int8",
    "sell_price": "float32",
    "item_id_mean_enc": "float32",
    "dept_id_mean_enc": "float32",
    "cat_id_mean_enc": "float32",
    "store_id_mean_enc": "float32",
    "state_id_mean_enc": "float32",
    "store_cat_mean_enc": "float32",
    "store_dept_mean_enc": "float32",
}


def partition_csv_gz_by_store(csv_path: Path, out_dir: Path, track_price_stats: bool = False) -> dict:
    """Stream a big .csv.gz file in chunks (pandas) and fan rows out into one
    Parquet file per store_id. Never holds more than one chunk in memory."""
    if out_dir.exists():
        for f in out_dir.glob("*.parquet"):
            f.unlink()
    out_dir.mkdir(parents=True, exist_ok=True)

    writers: dict[str, pq.ParquetWriter] = {}
    price_sum = 0.0
    price_count = 0
    total_rows = 0

    reader = pd.read_csv(
        csv_path,
        compression="gzip",
        chunksize=PARTITION_CHUNK_ROWS,
        dtype=PANDAS_DTYPE_MAP,
        low_memory=False,
    )
    for chunk_idx, chunk in enumerate(reader):
        total_rows += len(chunk)

        if track_price_stats and "sell_price" in chunk.columns:
            valid_prices = chunk["sell_price"].dropna()
            price_sum += float(valid_prices.sum())
            price_count += int(valid_prices.shape[0])

        for store, sub in chunk.groupby("store_id", sort=False):
            table = pa.Table.from_pandas(sub, preserve_index=False)
            if store not in writers:
                writers[store] = pq.ParquetWriter(out_dir / f"{store}.parquet", table.schema)
            writers[store].write_table(table)

        print(f"  chunk {chunk_idx}: {total_rows:,} rows processed so far", end="\r")
        del chunk
        gc.collect()

    print()
    for w in writers.values():
        w.close()

    result = {"stores": sorted(writers.keys()), "total_rows": total_rows}
    if track_price_stats:
        result["global_price_mean"] = (price_sum / price_count) if price_count else 0.0
    return result


In [4]:
print("Partitioning train file by store...")
train_partition_info = partition_csv_gz_by_store(TRAIN_INPUT_PATH, TRAIN_PARTITION_DIR, track_price_stats=True)

print("Partitioning test file by store...")
test_partition_info = partition_csv_gz_by_store(TEST_INPUT_PATH, TEST_PARTITION_DIR, track_price_stats=False)

STORES = train_partition_info["stores"]
GLOBAL_PRICE_MEAN = train_partition_info["global_price_mean"]

print("stores found:      ", STORES)
print("train total rows:  ", f"{train_partition_info['total_rows']:,}")
print("test total rows:   ", f"{test_partition_info['total_rows']:,}")
print("global_price_mean: ", GLOBAL_PRICE_MEAN)


Partitioning train file by store...
  chunk 29: 58,327,370 rows processed so far
Partitioning test file by store...
  chunk 0: 853,720 rows processed so far
stores found:       ['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']
train total rows:   58,327,370
test total rows:    853,720
global_price_mean:  4.446164778645086


## Fase 2 - helper functions (Polars, dijalankan per store)

Sinyal fungsional identik dengan versi sebelumnya, hanya `add_price_features` yang
sekarang menerima `global_price_mean` sebagai parameter (dihitung sekali di Fase 1)
alih-alih menghitungnya ulang dari `train_reference` tiap kali dipanggil.


In [5]:
def cast_numeric_columns(df: pl.DataFrame) -> pl.DataFrame:
    if df.height == 0:
        return df
    return df.with_columns(
        pl.col("date").str.to_date("%Y-%m-%d", strict=False)
    )


def add_calendar_features(df: pl.DataFrame) -> pl.DataFrame:
    out = df.with_columns([
        pl.col("date").dt.week().cast(pl.Int32).alias("week"),
        pl.col("date").dt.day().cast(pl.Int16).alias("day"),
    ])

    event_dates = (
        out.filter(
            (pl.col("event_name_1").cast(pl.Utf8) != "No_Event")
            | (pl.col("event_name_2").cast(pl.Utf8) != "No_Event")
        )
        .select("date")
        .unique()
        .sort("date")
    )

    if event_dates.height == 0:
        return out.with_columns(pl.lit(999).cast(pl.Int32).alias("days_to_event"))

    calendar = out.select("date").unique().sort("date")
    calendar = calendar.join_asof(
        event_dates.rename({"date": "next_event_date"}),
        left_on="date",
        right_on="next_event_date",
        strategy="forward",
    )
    calendar = calendar.with_columns(
        pl.when(pl.col("next_event_date").is_null())
        .then(999)
        .otherwise((pl.col("next_event_date") - pl.col("date")).dt.total_days())
        .cast(pl.Int32)
        .alias("days_to_event")
    ).select(["date", "days_to_event"])

    return out.join(calendar, on="date", how="left")


def _fill_inf(expr: pl.Expr) -> pl.Expr:
    return (
        pl.when(expr.is_infinite() | expr.is_nan())
        .then(0.0)
        .otherwise(expr)
        .fill_null(0.0)
    )


def add_price_features(df: pl.DataFrame, train_reference: pl.DataFrame, global_price_mean: float) -> tuple[pl.DataFrame, dict]:
    price_mean_map = (
        train_reference.group_by(["store_id", "item_id"])
        .agg(pl.col("sell_price").mean().alias("price_mean"))
    )

    out = df.join(price_mean_map, on=["store_id", "item_id"], how="left")
    out = out.with_columns(pl.col("price_mean").fill_null(global_price_mean))

    out = out.with_columns(
        (pl.col("sell_price") / pl.col("price_mean")).alias("price_norm")
    )
    out = out.with_columns(
        pl.col("sell_price").mean().over(["store_id", "cat_id", "wm_yr_wk"]).alias("price_cat_avg")
    )
    out = out.with_columns(
        (pl.col("sell_price") / pl.col("price_cat_avg")).alias("price_cat_ratio")
    )

    out = out.sort(["store_id", "item_id", "date"])
    out = out.with_columns(
        pl.col("sell_price").diff().over(["store_id", "item_id"]).alias("price_change")
    )
    out = out.with_columns([
        pl.col("price_change").fill_null(0).ne(0).cast(pl.Int8).alias("price_changed"),
        (pl.col("sell_price") < pl.col("price_mean")).cast(pl.Int8).alias("is_discounted"),
    ])

    out = out.with_columns([
        _fill_inf(pl.col("price_norm")).alias("price_norm"),
        _fill_inf(pl.col("price_cat_ratio")).alias("price_cat_ratio"),
        _fill_inf(pl.col("price_change")).alias("price_change"),
    ])

    artifact = {
        "price_reference_pairs": int(price_mean_map.height),
    }
    return out, artifact


def add_lag_features(df: pl.DataFrame, lag_days: list[int]) -> pl.DataFrame:
    lag_exprs = [
        pl.col("sales").shift(lag).over("id").fill_null(0).alias(f"lag_{lag}")
        for lag in lag_days
    ]
    return df.with_columns(lag_exprs)


def add_rolling_features(df: pl.DataFrame, windows: list[int]) -> pl.DataFrame:
    out = df.with_columns(
        pl.col("sales").shift(1).over("id").alias("_shifted_sales")
    )

    roll_exprs = []
    for window in windows:
        base = pl.col("_shifted_sales")
        roll_exprs.extend([
            base.rolling_mean(window_size=window, min_periods=1).over("id").alias(f"rolling_mean_{window}"),
            base.rolling_std(window_size=window, min_periods=1).over("id").alias(f"rolling_std_{window}"),
            base.rolling_max(window_size=window, min_periods=1).over("id").alias(f"rolling_max_{window}"),
            base.rolling_min(window_size=window, min_periods=1).over("id").alias(f"rolling_min_{window}"),
        ])
    out = out.with_columns(roll_exprs).drop("_shifted_sales")

    rolling_cols = [c for c in out.columns if c.startswith("rolling_")]
    out = out.with_columns([pl.col(c).fill_null(0) for c in rolling_cols])
    return out


def downcast_feature_columns(df: pl.DataFrame) -> pl.DataFrame:
    float_cols = [
        c for c in df.columns
        if c.startswith("lag_") or c.startswith("rolling_") or c.startswith("price_")
    ]
    exprs = [pl.col(c).cast(pl.Float32) for c in float_cols]
    if "days_to_event" in df.columns:
        exprs.append(pl.col("days_to_event").cast(pl.Int32))
    if "week" in df.columns:
        exprs.append(pl.col("week").cast(pl.Int16))
    if "day" in df.columns:
        exprs.append(pl.col("day").cast(pl.Int16))
    return df.with_columns(exprs)


## Fase 2 - loop per store: bangun fitur, lalu append ke satu file gabungan

Satu file gzip dibuka sekali sebelum loop dimulai dan ditutup sekali di akhir -
setiap store menulis baris CSV-nya ke stream gzip yang SAMA (bukan
`gzip.open(...)` baru per store), supaya hasil akhirnya tetap satu gzip member yang
valid dan bisa dibaca ulang oleh tool apa pun tanpa syarat khusus.

Sebagai bonus, hasil per-store juga ditulis ke `.parquet` (satu file per store) di
`TRAIN_OUTPUT_PARQUET_DIR` / `TEST_OUTPUT_PARQUET_DIR` - kalau nanti notebook
modeling butuh baca ulang dataset ini secara hemat memory (misalnya per store atau
per fold), baca dari sini, jangan dari `.csv.gz` gabungan.


In [6]:
TRAIN_OUTPUT_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
TEST_OUTPUT_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

train_gz = gzip.open(TRAIN_OUTPUT_PATH, "wt", newline="")
test_gz = gzip.open(TEST_OUTPUT_PATH, "wt", newline="")

train_header_written = False
test_header_written = False
train_rows_written = 0
test_rows_written = 0
price_artifacts = []
sanity_check_sample = None  # keep one small store's frame around for the sanity-check section below
expected_train_columns = None
expected_test_columns = None

try:
    for store in STORES:
        train_part_path = TRAIN_PARTITION_DIR / f"{store}.parquet"
        test_part_path = TEST_PARTITION_DIR / f"{store}.parquet"

        store_train = pl.read_parquet(train_part_path) if train_part_path.exists() else pl.DataFrame()
        store_test = pl.read_parquet(test_part_path) if test_part_path.exists() else pl.DataFrame()

        store_train = cast_numeric_columns(store_train)
        store_test = cast_numeric_columns(store_test)

        if store_test.height:
            combined = pl.concat([store_train, store_test], how="vertical_relaxed")
        else:
            combined = store_train
        combined = combined.sort(["id", "date"])

        combined = add_calendar_features(combined)
        combined, price_artifact = add_price_features(
            combined, train_reference=store_train, global_price_mean=GLOBAL_PRICE_MEAN
        )
        price_artifact["store_id"] = store
        price_artifacts.append(price_artifact)

        combined = add_lag_features(combined, LAG_DAYS)
        combined = add_rolling_features(combined, ROLLING_WINDOWS)
        combined = downcast_feature_columns(combined)

        store_train_out = combined.filter(pl.col("split") == "train")
        store_test_out = combined.filter(pl.col("split") == "test_public")

        # Guard: every store must produce the exact same column set/order, since only
        # the first store writes a header - a mismatch here would silently misalign
        # columns under the wrong header names for every store after the first.
        if expected_train_columns is None:
            expected_train_columns = store_train_out.columns
        else:
            assert store_train_out.columns == expected_train_columns, (
                f"Column mismatch for store {store}: {store_train_out.columns} != {expected_train_columns}"
            )

        store_train_out.write_csv(train_gz, include_header=not train_header_written)
        train_header_written = True
        train_rows_written += store_train_out.height
        store_train_out.write_parquet(TRAIN_OUTPUT_PARQUET_DIR / f"{store}.parquet")

        if store_test_out.height:
            if expected_test_columns is None:
                expected_test_columns = store_test_out.columns
            else:
                assert store_test_out.columns == expected_test_columns, (
                    f"Column mismatch for store {store}: {store_test_out.columns} != {expected_test_columns}"
                )
            store_test_out.write_csv(test_gz, include_header=not test_header_written)
            test_header_written = True
            test_rows_written += store_test_out.height
            store_test_out.write_parquet(TEST_OUTPUT_PARQUET_DIR / f"{store}.parquet")

        if sanity_check_sample is None:
            sanity_check_sample = combined.clone()

        print(f"[{store}] train rows: {store_train_out.height:,} | test rows: {store_test_out.height:,}")

        del store_train, store_test, combined, store_train_out, store_test_out
        gc.collect()
finally:
    train_gz.close()
    test_gz.close()

print()
print("total train rows written:", f"{train_rows_written:,}")
print("total test rows written: ", f"{test_rows_written:,}")

assert train_rows_written == train_partition_info["total_rows"], (
    f"Row count mismatch (train): wrote {train_rows_written:,}, "
    f"expected {train_partition_info['total_rows']:,} - some rows were lost or duplicated."
)
assert test_rows_written == test_partition_info["total_rows"], (
    f"Row count mismatch (test): wrote {test_rows_written:,}, "
    f"expected {test_partition_info['total_rows']:,} - some rows were lost or duplicated."
)
print("row-count reconciliation OK: every input row made it to the output exactly once.")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13100\1978325711.py:112: DeprecationWarning: the argument `min_periods` for `Expr.rolling_mean` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  base.rolling_mean(window_size=window, min_periods=1).over("id").alias(f"rolling_mean_{window}"),
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13100\1978325711.py:113: DeprecationWarning: the argument `min_periods` for `Expr.rolling_std` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  base.rolling_std(window_size=window, min_periods=1).over("id").alias(f"rolling_std_{window}"),
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13100\1978325711.py:114: DeprecationWarning: the argument `min_periods` for `Expr.rolling_max` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  base.rolling_max(window_size=window, min_periods=1).over("id").alias(f"rolling_max_{window}"),
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13100\1978325711.py:115: DeprecationWarning

[CA_1] train rows: 5,832,737 | test rows: 85,372
[CA_2] train rows: 5,832,737 | test rows: 85,372
[CA_3] train rows: 5,832,737 | test rows: 85,372
[CA_4] train rows: 5,832,737 | test rows: 85,372
[TX_1] train rows: 5,832,737 | test rows: 85,372
[TX_2] train rows: 5,832,737 | test rows: 85,372
[TX_3] train rows: 5,832,737 | test rows: 85,372
[WI_1] train rows: 5,832,737 | test rows: 85,372
[WI_2] train rows: 5,832,737 | test rows: 85,372
[WI_3] train rows: 5,832,737 | test rows: 85,372

total train rows written: 58,327,370
total test rows written:  853,720
row-count reconciliation OK: every input row made it to the output exactly once.


## Leakage sanity checks

Dijalankan pada frame satu store yang sudah dipegang di memory dari iterasi
pertama (`sanity_check_sample`) - jadi tidak perlu membaca ulang file gabungan yang
besar sama sekali.


In [7]:
assert sanity_check_sample is not None, "tidak ada store yang diproses"

first_id = sanity_check_sample.select("id").row(0)[0]
first_series = sanity_check_sample.filter(pl.col("id") == first_id).sort("date")

assert (first_series.head(7)["lag_7"] == 0).all(), "lag_7 pada burn-in period harus nol"
assert first_series.head(1)["rolling_mean_7"].item() == 0, "rolling_mean_7 awal harus nol"

train_only_for_corr = sanity_check_sample.filter(pl.col("split") == "train")
corr_same_day = train_only_for_corr.select(pl.corr("sales", "rolling_mean_7")).item()
corr_prev_day = train_only_for_corr.select(pl.corr("lag_7", "rolling_mean_7")).item()

print("checked on store:", first_series.select("store_id").row(0)[0])
print("corr(sales, rolling_mean_7):", corr_same_day)
print("corr(lag_7, rolling_mean_7):", corr_prev_day)


checked on store: CA_1
corr(sales, rolling_mean_7): 0.8062469932561357
corr(lag_7, rolling_mean_7): 0.8509663939476013


## Ringkasan akhir + bersihkan file partisi sementara


In [8]:
import shutil

feature_summary = {
    "source": "Outline Skripsi Bab 3 - Tahap Feature Engineering",
    "mode": MODE,
    "compression": COMPRESSION,
    "engine": "polars (chunked per-store; pandas used only for the streaming gzip partition pass)",
    "stores_processed": STORES,
    "partition_chunk_rows": PARTITION_CHUNK_ROWS,
    "lag_days": LAG_DAYS,
    "rolling_windows": ROLLING_WINDOWS,
    "rolling_statistics": ROLLING_STATS,
    "notes": [
        "Lag features dihitung dengan shift(k) sehingga hanya memakai informasi masa lalu.",
        "Rolling statistics dihitung dari shift(1) sebelum rolling untuk mencegah lookahead leakage.",
        "price_mean dihitung per store dari train_reference store tersebut; global_price_mean (dihitung sekali di Fase 1 dari seluruh train) dipakai sebagai fallback.",
        "Feature engineering dijalankan per store untuk menjaga peak memory tetap rendah pada mesin dengan RAM terbatas (~10 GB usable).",
        "File .csv.gz akhir tetap satu dataset gabungan (semua store); versi .parquet per-store juga disediakan untuk pembacaan yang lebih hemat memory di notebook berikutnya.",
    ],
    "global_price_mean": GLOBAL_PRICE_MEAN,
    "price_artifacts_by_store": price_artifacts,
    "train_result": {
        "row_count": train_rows_written,
        "output_path": str(TRAIN_OUTPUT_PATH.resolve()),
        "parquet_by_store_dir": str(TRAIN_OUTPUT_PARQUET_DIR.resolve()),
    },
    "test_result": {
        "row_count": test_rows_written,
        "output_path": str(TEST_OUTPUT_PATH.resolve()),
        "parquet_by_store_dir": str(TEST_OUTPUT_PARQUET_DIR.resolve()),
    },
}

with open(SUMMARY_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_summary, f, indent=2)

# clean up the temporary per-store partition parquet files from Fase 1 (not the final output)
shutil.rmtree(TRAIN_PARTITION_DIR, ignore_errors=True)
shutil.rmtree(TEST_PARTITION_DIR, ignore_errors=True)

feature_summary


{'source': 'Outline Skripsi Bab 3 - Tahap Feature Engineering',
 'mode': 'full',
 'compression': 'gzip',
 'engine': 'polars (chunked per-store; pandas used only for the streaming gzip partition pass)',
 'stores_processed': ['CA_1',
  'CA_2',
  'CA_3',
  'CA_4',
  'TX_1',
  'TX_2',
  'TX_3',
  'WI_1',
  'WI_2',
  'WI_3'],
 'partition_chunk_rows': 2000000,
 'lag_days': [7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 90, 180, 364],
 'rolling_windows': [7, 14, 28, 60, 90, 180, 364],
 'rolling_statistics': ['mean', 'std', 'max', 'min'],
 'notes': ['Lag features dihitung dengan shift(k) sehingga hanya memakai informasi masa lalu.',
  'Rolling statistics dihitung dari shift(1) sebelum rolling untuk mencegah lookahead leakage.',
  'price_mean dihitung per store dari train_reference store tersebut; global_price_mean (dihitung sekali di Fase 1 dari seluruh train) dipakai sebagai fallback.',
  'Feature engineering dijalankan per store untuk menjaga peak memory tetap rendah pada mesin dengan RAM t

## Preview hasil akhir (baca sedikit baris saja, bukan seluruh file)


In [9]:
train_preview = pl.read_csv(TRAIN_OUTPUT_PATH, n_rows=5)
test_preview = pl.read_csv(TEST_OUTPUT_PATH, n_rows=5)

print("train rows written:", f"{train_rows_written:,}")
print("test rows written: ", f"{test_rows_written:,}")
train_preview.select(["id", "date", "sales", "lag_7", "lag_28", "rolling_mean_7", "rolling_std_28", "price_norm", "days_to_event"])


train rows written: 58,327,370
test rows written:  853,720


id,date,sales,lag_7,lag_28,rolling_mean_7,rolling_std_28,price_norm,days_to_event
str,str,i64,f64,f64,f64,f64,f64,i64
"""FOODS_1_001_CA_1_evaluation""","""2011-01-29""",3,0.0,0.0,0.0,0.0,0.9228884,8
"""FOODS_1_001_CA_1_evaluation""","""2011-01-30""",0,0.0,0.0,3.0,0.0,0.9228884,7
"""FOODS_1_001_CA_1_evaluation""","""2011-01-31""",0,0.0,0.0,1.5,2.1213202,0.9228884,6
"""FOODS_1_001_CA_1_evaluation""","""2011-02-01""",1,0.0,0.0,1.0,1.7320508,0.9228884,5
"""FOODS_1_001_CA_1_evaluation""","""2011-02-02""",4,0.0,0.0,1.0,1.4142135,0.9228884,4
